In [357]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [358]:
df=pd.read_parquet('../../data/solar-flare-events/solar_flare_events.parquet')
df.head()


,satellite,start_time,peak_time,peak_flux_wm2,goes_class,end_time,goes_class_letter
0,GOES-16,2017-02-09 00:41:00,2017-02-09 00:50:00,3.711663e-07,B3.7,2017-02-09 00:56:00,B
1,GOES-16,2017-02-09 01:30:00,2017-02-09 01:40:00,4.310535e-07,B4.3,2017-02-09 01:42:00,B
2,GOES-16,2017-02-09 01:45:00,2017-02-09 01:51:00,1.694939e-06,C1.6,2017-02-09 01:55:00,C
3,GOES-16,2017-02-09 02:31:00,2017-02-09 02:40:00,3.234027e-07,B3.2,2017-02-09 02:48:00,B
4,GOES-16,2017-02-09 02:55:00,2017-02-09 02:59:00,3.839661e-07,B3.8,NaT,B


In [359]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16113 entries, 0 to 16112
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   satellite          16113 non-null  string        
 1   start_time         16113 non-null  datetime64[us]
 2   peak_time          16087 non-null  datetime64[us]
 3   peak_flux_wm2      16087 non-null  float64       
 4   goes_class         16087 non-null  string        
 5   end_time           13971 non-null  datetime64[us]
 6   goes_class_letter  16087 non-null  string        
dtypes: datetime64[us](3), float64(1), string(3)
memory usage: 1.0 MB


In [360]:
df.describe()

,start_time,peak_time,peak_flux_wm2,end_time
count,16113,16087,1.608700e+04,13971
mean,2022-06-14 13:09:43.198659,2022-06-14 05:58:05.329769,6.000592e-06,2022-06-15 01:24:04.539403
min,2017-02-09 00:41:00,2017-02-09 00:50:00,9.748320e-08,2017-02-09 00:56:00
25%,2021-08-27 07:10:00,2021-08-27 00:47:00,7.210172e-07,2021-08-26 18:52:30
50%,2022-12-16 04:03:00,2022-12-15 22:18:00,2.330065e-06,2022-12-14 22:16:00
75%,2024-02-24 08:03:00,2024-02-24 02:30:00,5.000198e-06,2024-02-22 09:10:30
max,2026-06-10 08:08:00,2026-06-10 09:09:00,1.464082e-03,2026-06-10 03:36:00
std,NaN,NaN,2.677224e-05,NaN


In [361]:
print(f'Rows missing both peak_time and peak_flux_wm2: {len(df.query('peak_time.isnull() and peak_flux_wm2.isnull()'))}')

Rows missing both peak_time and peak_flux_wm2: 26


In [362]:
print(f"Rows missing peak_time but not peak_flux_wm2: {len(df.query('peak_time.isnull() and not peak_flux_wm2.isnull()'))}")
print(f"Rows missing peak_flux_wm2 but not peak_time: {len(df.query('not peak_time.isnull() and peak_flux_wm2.isnull()'))}")

Rows missing peak_time but not peak_flux_wm2: 0
Rows missing peak_flux_wm2 but not peak_time: 0


In [363]:
df.query('peak_time.isnull() and peak_flux_wm2.isnull()')

,satellite,start_time,peak_time,peak_flux_wm2,goes_class,end_time,goes_class_letter
131,GOES-16,2017-03-04 05:30:00,NaT,NaN,<NA>,NaT,<NA>
267,GOES-16,2017-04-03 05:16:00,NaT,NaN,<NA>,NaT,<NA>
1205,GOES-16,2017-08-31 05:28:00,NaT,NaN,<NA>,NaT,<NA>
1415,GOES-16,2017-09-26 04:59:00,NaT,NaN,<NA>,NaT,<NA>
4120,GOES-16,2021-09-04 04:17:00,NaT,NaN,<NA>,NaT,<NA>
5662,GOES-16,2022-03-22 04:07:00,NaT,NaN,<NA>,NaT,<NA>
7098,GOES-16,2022-08-30 04:44:00,NaT,NaN,<NA>,NaT,<NA>
7238,GOES-16,2022-09-12 04:12:00,NaT,NaN,<NA>,NaT,<NA>
9085,GOES-16,2023-04-13 04:41:00,NaT,NaN,<NA>,NaT,<NA>
9267,GOES-16,2023-05-03 12:05:00,NaT,NaN,<NA>,NaT,<NA>


We know almost nothing about these events except their start time, so the missing values can't be inferred. Dropping the corresponding rows.

In [364]:
df=df.drop(index=df[df.peak_time.isnull() & df.peak_flux_wm2.isnull()].index)
df.info()


<class 'pandas.DataFrame'>
Index: 16087 entries, 0 to 16112
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   satellite          16087 non-null  string        
 1   start_time         16087 non-null  datetime64[us]
 2   peak_time          16087 non-null  datetime64[us]
 3   peak_flux_wm2      16087 non-null  float64       
 4   goes_class         16087 non-null  string        
 5   end_time           13971 non-null  datetime64[us]
 6   goes_class_letter  16087 non-null  string        
dtypes: datetime64[us](3), float64(1), string(3)
memory usage: 1.2 MB


In [365]:
# Checking which categories of events are missing their end time
df[df.end_time.isnull()]['goes_class_letter'].value_counts()

goes_class_letter
C    1315
B     642
M     158
X       1
Name: count, dtype: int64[pyarrow]

In [366]:
percents=((df[df.end_time.isnull()]['goes_class_letter'].value_counts().values)/len(df)*100).round(2)
print("Percent of missing end time by class")
for c, p in zip(df[df.end_time.isnull()]['goes_class_letter'].value_counts().index, percents):
    print(f"{c}: {p}%")

Percent of missing end time by class
C: 8.17%
B: 3.99%
M: 0.98%
X: 0.01%


In [367]:
null_end_time=(df.end_time.isnull())
df[null_end_time]

,satellite,start_time,peak_time,peak_flux_wm2,goes_class,end_time,goes_class_letter
4,GOES-16,2017-02-09 02:55:00,2017-02-09 02:59:00,3.839661e-07,B3.8,NaT,B
15,GOES-16,2017-02-09 10:53:00,2017-02-09 11:00:00,1.166238e-07,B1.1,NaT,B
17,GOES-16,2017-02-09 11:52:00,2017-02-09 12:04:00,2.097050e-07,B2.0,NaT,B
28,GOES-16,2017-02-19 08:58:00,2017-02-19 09:06:00,4.473251e-07,B4.4,NaT,B
60,GOES-16,2017-02-22 17:20:00,2017-02-22 17:27:00,1.595764e-07,B1.5,NaT,B
...,...,...,...,...,...,...,...
16065,GOES-16,2025-04-01 06:25:00,2025-04-01 06:35:00,4.762585e-06,C4.7,NaT,C
16073,GOES-16,2025-04-01 20:45:00,2025-04-01 21:00:00,5.220541e-06,C5.2,NaT,C
16075,GOES-16,2025-04-01 22:00:00,2025-04-01 22:08:00,3.569627e-06,C3.5,NaT,C
16077,GOES-16,2025-04-02 01:15:00,2025-04-02 01:26:00,2.308811e-06,C2.3,NaT,C


In [368]:
df['duration_minutes'] = (df['end_time'] - df['start_time']).dt.total_seconds() / 60
df


,satellite,start_time,peak_time,peak_flux_wm2,goes_class,end_time,goes_class_letter,duration_minutes
0,GOES-16,2017-02-09 00:41:00,2017-02-09 00:50:00,3.711663e-07,B3.7,2017-02-09 00:56:00,B,15.0
1,GOES-16,2017-02-09 01:30:00,2017-02-09 01:40:00,4.310535e-07,B4.3,2017-02-09 01:42:00,B,12.0
2,GOES-16,2017-02-09 01:45:00,2017-02-09 01:51:00,1.694939e-06,C1.6,2017-02-09 01:55:00,C,10.0
3,GOES-16,2017-02-09 02:31:00,2017-02-09 02:40:00,3.234027e-07,B3.2,2017-02-09 02:48:00,B,17.0
4,GOES-16,2017-02-09 02:55:00,2017-02-09 02:59:00,3.839661e-07,B3.8,NaT,B,NaN
...,...,...,...,...,...,...,...,...
16108,GOES-16,2025-04-06 15:46:00,2025-04-06 15:53:00,2.960594e-06,C2.9,2025-04-06 16:05:00,C,19.0
16109,GOES-16,2025-04-06 17:41:00,2025-04-06 17:44:00,2.088433e-06,C2.0,2025-04-06 17:47:00,C,6.0
16110,GOES-16,2025-04-06 19:47:00,2025-04-06 19:57:00,2.385995e-06,C2.3,2025-04-06 20:06:00,C,19.0
16111,GOES-18,2026-06-10 01:01:00,2026-06-10 09:09:00,9.800000e-04,C2.6,2026-06-10 00:01:00,C,-60.0


In [369]:
negative_duration=df.query('duration_minutes<0')
negative_duration

,satellite,start_time,peak_time,peak_flux_wm2,goes_class,end_time,goes_class_letter,duration_minutes
16111,GOES-18,2026-06-10 01:01:00,2026-06-10 09:09:00,0.00098,C2.6,2026-06-10 00:01:00,C,-60.0
16112,GOES-18,2026-06-10 08:08:00,2026-06-10 03:03:00,0.00055,C1.0,2026-06-10 03:36:00,C,-272.0


In [370]:
df.drop(negative_duration.index, inplace=True)

In [ ]:
# Keep only rows with a known end time and class, compute duration in minutes,
# and drop the 2 bad rows where end_time is earlier than start_time.
flare_order = ['A', 'B', 'C', 'M', 'X']
plot_df = df.dropna(subset=['end_time', 'goes_class_letter']).copy()
plot_df['duration_minutes'] = (
    (plot_df['end_time'] - plot_df['start_time']).dt.total_seconds() / 60
)
plot_df = plot_df.query('duration_minutes >= 0')

# Group durations by flare class in ladder order (A weakest -> X strongest),
# skipping classes with no data.
classes = [c for c in flare_order if c in plot_df['goes_class_letter'].values]
durations_by_class = [
    plot_df.loc[plot_df['goes_class_letter'] == c, 'duration_minutes']
    for c in classes
]

# Draw a box plot of flare duration per class with a dark cosmic style
plt.style.use('dark_background')
cosmic_colors = ['#b388ff', '#7c4dff', '#40c4ff', '#ff80ab', '#ffd740']

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0b0c2a')
ax.set_facecolor('#0b0c2a')

boxes = ax.boxplot(
    durations_by_class,
    tick_labels=classes,
    patch_artist=True,
    medianprops={'color': 'white', 'linewidth': 1.5},
    flierprops={'marker': '.', 'markersize': 3, 'markerfacecolor': '#e0e0ff',
                'markeredgecolor': 'none', 'alpha': 0.5},
)
for patch, color in zip(boxes['boxes'], cosmic_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
    patch.set_edgecolor(color)

ax.set_title('Solar Flare Duration by GOES Class', color='#e0e0ff', fontsize=14)
ax.set_xlabel('GOES class (weakest → strongest)', color='#9fa8da')
ax.set_ylabel('Duration (minutes)', color='#9fa8da')
ax.grid(axis='y', color='#3949ab', alpha=0.3, linewidth=0.5)
fig.savefig('outputs/flare_duration_by_class_boxplot.png', dpi=300,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

I looked at flare duration within each intensity class. Stronger flares last longer: the median duration grows from 9 minutes for A up to 25 minutes for X. Every class also has a tail of unusually long flares (the dots above the whiskers), so the class medians are a safer "typical value" than the means.

# Imputing strategy for end time and duration
I am going to impute the missing duration with the median (not the mean, due to outliers in each class) of events in the same intensity category (A, B, C, M, or X), since a flare's duration is correlated with its intensity. The fraction of missing end times is fairly small, so median imputation won't degrade data quality much.
I will also add a binary `end_time_imputed` column to give the model a signal for which rows were filled, then use the imputed duration to reconstruct the `end_time` column.

In [372]:
mean_duration_by_class=df.groupby('goes_class_letter')['duration_minutes'].median()
mean_duration_by_class

goes_class_letter
A     9.0
B    13.0
C    14.0
M    19.0
X    25.0
Name: duration_minutes, dtype: float64

In [373]:
# Flag which rows are about to be imputed
df['end_time_imputed'] = df['end_time'].isna().astype(int)

# Fill duration_minutes with class median
df['duration_minutes'] = df['duration_minutes'].fillna(
    df.groupby('goes_class_letter')['duration_minutes'].transform('median')
)

# Reconstruct end_time from start_time + filled duration
df['end_time'] = df['end_time'].fillna(
    df['start_time'] + pd.to_timedelta(df['duration_minutes'], unit='m')
)

In [374]:
df.isnull().sum()

satellite            0
start_time           0
peak_time            0
peak_flux_wm2        0
goes_class           0
end_time             0
goes_class_letter    0
duration_minutes     0
end_time_imputed     0
dtype: int64

Next: look at the overall distribution of duration, a histogram of duration by class, and the relationship between `peak_flux_wm2` and duration (colored by class).

In [375]:
# Shared cosmic style: one color per flare class, reused in the next cells
plt.style.use('dark_background')
SPACE_BG = '#0b0c2a'
CLASS_COLORS = {'A': '#b388ff', 'B': '#7c4dff', 'C': '#40c4ff',
                'M': '#ff80ab', 'X': '#ffd740'}

In [ ]:
# Histogram of flare duration across all classes, with median and mean markers
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(SPACE_BG)
ax.set_facecolor(SPACE_BG)

ax.hist(plot_df['duration_minutes'], bins=60, color='#7c4dff',
        edgecolor='#0b0c2a', alpha=0.85)
ax.axvline(plot_df['duration_minutes'].median(), color='#ffd740',
           linewidth=1.5, label=f"median = {plot_df['duration_minutes'].median():.0f} min")
ax.axvline(plot_df['duration_minutes'].mean(), color='#ff80ab',
           linewidth=1.5, linestyle='--',
           label=f"mean = {plot_df['duration_minutes'].mean():.1f} min")

ax.set_title('Solar Flare Duration — Overall Distribution', color='#e0e0ff', fontsize=14)
ax.set_xlabel('Duration (minutes)', color='#9fa8da')
ax.set_ylabel('Number of flares', color='#9fa8da')
ax.legend(facecolor=SPACE_BG, edgecolor='#3949ab')
ax.grid(axis='y', color='#3949ab', alpha=0.3, linewidth=0.5)
fig.savefig('outputs/flare_duration_distribution.png', dpi=300,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

I looked at the overall distribution of duration. Most flares are over within about 20 minutes, but there is a long right tail of rare drawn-out events that pulls the mean above the median. This skew is the reason I filled missing durations with the median.

In [ ]:
# Overlaid density histograms of duration per class (A is skipped: only 2 events).
# density=True puts the unbalanced classes (9.6k C vs 82 X) on a comparable scale.
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(SPACE_BG)
ax.set_facecolor(SPACE_BG)

bins = np.arange(0, plot_df['duration_minutes'].max() + 5, 5)
for cls in ['B', 'C', 'M', 'X']:
    subset = plot_df.loc[plot_df['goes_class_letter'] == cls, 'duration_minutes']
    ax.hist(subset, bins=bins, density=True, histtype='step',
            linewidth=2, color=CLASS_COLORS[cls],
            label=f'{cls} (n={len(subset):,})')

ax.set_title('Flare Duration by GOES Class', color='#e0e0ff', fontsize=14)
ax.set_xlabel('Duration (minutes)', color='#9fa8da')
ax.set_ylabel('Density', color='#9fa8da')
ax.set_xlim(0, 120)
ax.legend(facecolor=SPACE_BG, edgecolor='#3949ab', title='GOES class')
ax.grid(color='#3949ab', alpha=0.3, linewidth=0.5)
fig.savefig('outputs/flare_duration_by_class_density.png', dpi=300,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

I compared duration distributions between classes (as densities, since the classes are very unequal in size — 9.6k C-flares vs 82 X-flares). The pattern from the box plot holds: weaker classes peak at short durations, while M and X spread further to the right. So intensity class carries real information about how long a flare lasts.

In [ ]:
# Scatter of peak flux vs duration with class hue. Flux spans several orders of
# magnitude (B ~1e-7 to X ~1e-4 W/m^2), so the x-axis is log-scaled and the
# correlation is computed with Spearman (rank-based, robust to skew).
from scipy.stats import spearmanr

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(SPACE_BG)
ax.set_facecolor(SPACE_BG)

for cls in ['A', 'B', 'C', 'M', 'X']:
    subset = plot_df[plot_df['goes_class_letter'] == cls]
    ax.scatter(subset['peak_flux_wm2'], subset['duration_minutes'],
               s=8, alpha=0.4, color=CLASS_COLORS[cls], label=cls,
               edgecolors='none')

rho, _ = spearmanr(plot_df['peak_flux_wm2'], plot_df['duration_minutes'])
ax.set_xscale('log')
ax.set_title(f'Peak Flux vs Duration (Spearman ρ = {rho:.2f})',
             color='#e0e0ff', fontsize=14)
ax.set_xlabel('Peak flux (W/m², log scale)', color='#9fa8da')
ax.set_ylabel('Duration (minutes)', color='#9fa8da')
ax.legend(facecolor=SPACE_BG, edgecolor='#3949ab', title='GOES class',
          markerscale=2)
ax.grid(color='#3949ab', alpha=0.3, linewidth=0.5)
fig.savefig('outputs/peak_flux_vs_duration_scatter.png', dpi=300,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

I checked how peak flux relates to duration. There is a weak positive relationship — brighter flares tend to last somewhat longer — but the spread is large, so flux alone does not determine duration. The colored bands don't overlap on the x-axis simply because flux is what defines the class letter.

# Looking at outliers
Checking duration and flux for extreme values: first overall (beyond 3 standard deviations), then within each class using the IQR rule.

In [379]:
mean_duration=df.duration_minutes.mean()
std=df.duration_minutes.std()
outliers=df[(df.duration_minutes>mean_duration+3*std) | (df.duration_minutes<mean_duration-3*std)]
outliers

,satellite,start_time,peak_time,peak_flux_wm2,goes_class,end_time,goes_class_letter,duration_minutes,end_time_imputed
79,GOES-16,2017-02-24 16:57:00,2017-02-24 17:34:00,4.584236e-07,B4.5,2017-02-24 17:51:00,B,54.0,0
239,GOES-16,2017-04-01 19:21:00,2017-04-01 19:56:00,5.773486e-06,C5.7,2017-04-01 20:13:00,C,52.0,0
374,GOES-16,2017-04-17 02:11:00,2017-04-17 02:47:00,3.117804e-06,C3.1,2017-04-17 03:10:00,C,59.0,0
392,GOES-16,2017-04-18 19:15:00,2017-04-18 20:10:00,8.453525e-06,C8.4,2017-04-18 20:49:00,C,94.0,0
472,GOES-16,2017-05-27 17:59:00,2017-05-27 18:30:00,1.404303e-06,C1.4,2017-05-27 19:02:00,C,63.0,0
...,...,...,...,...,...,...,...,...,...
15804,GOES-16,2025-02-25 11:20:00,2025-02-25 11:59:00,3.636333e-05,M3.6,2025-02-25 12:43:00,M,83.0,0
15887,GOES-16,2025-03-08 17:38:00,2025-03-08 18:25:00,3.990003e-06,C3.9,2025-03-08 18:59:00,C,81.0,0
15893,GOES-16,2025-03-10 20:00:00,2025-03-10 20:32:00,2.082075e-06,C2.0,2025-03-10 20:56:00,C,56.0,0
15959,GOES-16,2025-03-20 16:49:00,2025-03-20 17:10:00,4.424483e-06,C4.4,2025-03-20 17:49:00,C,60.0,0


In [ ]:
# Flag IQR outliers within each group: a value is an outlier if it sits below
# Q1 - 1.5*IQR or above Q3 + 1.5*IQR of its own flare class.
def flag_group_outliers(data, value_col, group_col):
    grouped = data.groupby(group_col)[value_col]
    q1 = grouped.transform(lambda s: s.quantile(0.25))
    q3 = grouped.transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    return (data[value_col] < q1 - 1.5 * iqr) | (data[value_col] > q3 + 1.5 * iqr)


# Work on a log10 scale for flux: it spans orders of magnitude and the A->X
# class ladder is itself defined in powers of ten.
df['log_peak_flux'] = np.log10(df['peak_flux_wm2'])

# Flag per-class outliers separately for duration and for log peak flux.
df['duration_outlier'] = flag_group_outliers(df, 'duration_minutes', 'goes_class_letter')
df['flux_outlier'] = flag_group_outliers(df, 'log_peak_flux', 'goes_class_letter')

# Per-class summary: event count, outliers of each kind, and their rate.
flare_order = ['A', 'B', 'C', 'M', 'X']
outlier_summary = df.groupby('goes_class_letter').agg(
    n_events=('peak_flux_wm2', 'size'),
    duration_outliers=('duration_outlier', 'sum'),
    flux_outliers=('flux_outlier', 'sum'),
).reindex([c for c in flare_order if c in df['goes_class_letter'].values])
outlier_summary['duration_outlier_pct'] = (
    outlier_summary['duration_outliers'] / outlier_summary['n_events'] * 100
).round(2)
outlier_summary['flux_outlier_pct'] = (
    outlier_summary['flux_outliers'] / outlier_summary['n_events'] * 100
).round(2)
outlier_summary

In [ ]:
# Box plot of log10 peak flux per class to see flux spread and outliers
# (dots beyond the whiskers) within each intensity category.
classes = [c for c in flare_order if c in df['goes_class_letter'].values]
flux_by_class = [df.loc[df['goes_class_letter'] == c, 'log_peak_flux'] for c in classes]

plt.style.use('dark_background')
cosmic_colors = ['#b388ff', '#7c4dff', '#40c4ff', '#ff80ab', '#ffd740']

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0b0c2a')
ax.set_facecolor('#0b0c2a')

boxes = ax.boxplot(
    flux_by_class,
    tick_labels=classes,
    patch_artist=True,
    medianprops={'color': 'white', 'linewidth': 1.5},
    flierprops={'marker': '.', 'markersize': 3, 'markerfacecolor': '#e0e0ff',
                'markeredgecolor': 'none', 'alpha': 0.5},
)
for patch, color in zip(boxes['boxes'], cosmic_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
    patch.set_edgecolor(color)

ax.set_title('Solar Flare log10 Peak Flux by GOES Class', color='#e0e0ff', fontsize=14)
ax.set_xlabel('GOES class (weakest → strongest)', color='#9fa8da')
ax.set_ylabel('log10 peak flux (W/m²)', color='#9fa8da')
ax.grid(axis='y', color='#3949ab', alpha=0.3, linewidth=0.5)
fig.savefig('outputs/peak_flux_by_class_boxplot.png', dpi=300,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

I looked at peak flux within each class on a log10 scale. Each class sits roughly 10× above the previous one with almost no overlap — the class letter is essentially binned peak flux. Flux outliers only appear in M (1.3%) and X (6.1%): a few exceptionally bright events at the top of the scale, while duration outliers are spread evenly (~6–7%) across all classes.

In [382]:
df['satellite'].unique()

<ArrowStringArray>
['GOES-16']
Length: 1, dtype: string